# Section 5: From Latent Design to a Robot That Moves

The last step of the pipeline: take an optimized latent design, turn it into a standalone
MuJoCo model, and watch it execute the target human motion.

**Source**: `notebook/ae_test/mr_lego_{easy,middle,hard}_with_postrigging.ipynb`, `shared/voo_helpers.py`
**Paper**: arXiv:2604.08636 §III-C (Motion Retargeting and Procrustes Analysis),
§IV-A Eq. (5) $M^* = \mathcal{R}(x^*; M_{src})$, §V-E / Fig. 4-5 (hardware prototypes)

## What you get
1. Decode an optimized latent $z^*$ into a kinematic structure and see where it sits among the 30 robots
2. Export that structure as a **self-contained MJCF** (no mesh files) for simulation or hardware design
3. Retarget a CMU BVH motion onto it and recompute the paper's objective (PA-MPJPE + λ·N_joint)
4. Compare human vs. robot key poses and hand trajectories (Fig. 5 style)
5. Render the robot executing the motion
6. Plug in **your own BVH motion**
7. Use an **SMPL motion** (e.g. generated by MDM) instead of a BVH

## Safety
- `SAVE_OUTPUTS=False` default: nothing is written
- With `SAVE_OUTPUTS=True`, the MJCF / mp4 / png go to `./outputs/section5/` only

In [ ]:
# ---- Tutorial flags ----
SAVE_OUTPUTS = False      # if True, write MJCF + mp4 + png to ./outputs/section5/

MOTION_SOURCE = 'cmu'     # 'cmu' (BVH, default) | 'smpl' (SMPL motion, e.g. from MDM; see §7)
DESIGN_LATENT = None      # None → paper's optimum for MOTION, or e.g. [6.28, 0.57]
                          # (or the best latent printed at the end of 02_design_optimization_voo)

# with MOTION_SOURCE = 'cmu'
MOTION = 'easy'           # 'easy' (Wave Hello) | 'middle' (Chicken Dance) | 'hard' (Swimming)
CUSTOM_BVH = None         # None → the paper's CMU file for MOTION, or a path to your own .bvh (see §6)

# with MOTION_SOURCE = 'smpl' (paths are relative to notebook/ae_test, like the BVH paths)
SMPL_MOTION = '../../data/mdm_motion/sample00_rep00_smpl_params.npy'   # MDM *_smpl_params.npy (see §7)
SMPL_MODEL  = '../../data/smpl_model/model.pkl'                        # SMPL body model (see §7)

# ---- Path setup ----
import os, sys
HERE = os.path.abspath('.')
REPO = os.path.abspath('../../')
for p in ['package','package/yoon','package/mujoco_helper','package/kinematics_helper',
          'package/utility','package/bvh_helper','package/smpl','package/motion_retarget',
          'package/irvae_tools','package/kinematics','asset']:
    sys.path.insert(0, os.path.join(REPO, p))
sys.path.insert(0, HERE)

CANONICAL    = os.path.join(HERE, 'canonical/')
TUTORIAL_OUT = os.path.join(HERE, 'outputs/section5/')
if SAVE_OUTPUTS:
    os.makedirs(TUTORIAL_OUT, exist_ok=True)

# RobotCharacteristics uses ../../asset/_robot_info/A-pose.json (relative to notebook/ae_test)
os.chdir(os.path.join(REPO, 'notebook/ae_test'))

import time
import numpy as np
import torch as th
import mujoco
import matplotlib.pyplot as plt

from data_utils import load_saved_model, load_data_subset
from clustering_robots import RobotCharacteristics
from mujoco_parser import MuJoCoParserClass
from motion_retarget import directional_preretarget_postrigging, get_joi_body_name_of_irae
from bvh_parser import get_chains_from_bvh_cmu_mocap
from smpl_parser import get_chains_from_smpl
from smpl_model import SMPLModelClass
from shared.voo_helpers import (
    override_joi, evaluate, get_cmu_pos, get_smpl_pos,
    _build_envs_from_decoded, _sequence_pa_mpjpe, _similarity_transform_umeyama,
)

print(f'SAVE_OUTPUTS = {SAVE_OUTPUTS}, MOTION_SOURCE = {MOTION_SOURCE}, '
      f'MOTION = {SMPL_MOTION if MOTION_SOURCE == "smpl" else MOTION}')

## 1. Load the AutoEncoder and pick a design

`x_canonical` are the best latents found by VOO in the paper (Table I, screw + 2D IsoAE).
Their objective values are the reference numbers for §3.

In [ ]:
repr_kind  = 'screw'
model_name = 'h64_32_z2_isoTrue_actvTanh_isoreg1e-07_nepoch1000_bs10'
ae, _ = load_saved_model(model_name=model_name, device='cpu')

rc = RobotCharacteristics()
rc.max_joints_per_group = np.load(os.path.join(CANONICAL, 'data', repr_kind, 'njnt.npy'))

MOTIONS_BVH = {
    'easy':   '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/141/141_16.bvh',  # Wave Hello
    'middle': '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/018/18_15.bvh',   # Chicken Dance
    'hard':   '../../bvh/cmu_mocap_full_dataset/cmu_bvh/data/079/79_02.bvh',   # Swimming
}
x_canonical = {   # paper optimum per motion
    'easy':   np.array([6.28252373, 0.56721227]),
    'middle': np.array([4.69008544, 0.12768499]),
    'hard':   np.array([8.88798091, -4.74839331]),
}
reference_objective = {'easy': 74.5474, 'middle': 91.2086, 'hard': 96.5576}

z_star   = np.asarray(DESIGN_LATENT if DESIGN_LATENT is not None else x_canonical[MOTION], dtype=float)
bvh_path = CUSTOM_BVH if CUSTOM_BVH is not None else MOTIONS_BVH[MOTION]
assert MOTION_SOURCE in ('cmu', 'smpl'), MOTION_SOURCE
motion_path = SMPL_MOTION if MOTION_SOURCE == 'smpl' else bvh_path
motion_label = os.path.basename(SMPL_MOTION).replace('_smpl_params.npy', '') if MOTION_SOURCE == 'smpl' else MOTION
if MOTION_SOURCE == 'smpl':
    for p in (SMPL_MODEL, SMPL_MOTION):
        if not os.path.exists(p):
            raise FileNotFoundError(f'{p} not found: see §7 for how to obtain the SMPL model and motions')
print(f'design latent z* = {z_star}')
print(f'motion           = {motion_path} ({MOTION_SOURCE})')

# Where does z* sit among the 30 training robots?
data   = load_data_subset(os.path.join(CANONICAL, 'data', repr_kind) + '/')
labels = np.load(os.path.join(CANONICAL, 'data', repr_kind, 'label.npy'))
with th.no_grad():
    z_all = ae.encode(th.tensor(data, dtype=th.float32)).numpy()

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(z_all[:, 0], z_all[:, 1], s=40, c='lightgray', edgecolors='k', linewidths=0.5)
for i, name in enumerate(labels):
    ax.annotate(str(name), z_all[i], fontsize=7, alpha=0.7, xytext=(3, 3), textcoords='offset points')
ax.scatter(*z_star, marker='*', s=400, c='gold', edgecolors='k', zorder=5, label='design z*')
nearest = labels[np.argsort(np.linalg.norm(z_all - z_star, axis=1))[:3]]
ax.set_title(f'Fig 5.1: design z* in latent space (nearest: {", ".join(nearest)})')
ax.set_xlabel('z₀'); ax.set_ylabel('z₁'); ax.set_aspect('equal'); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 2. Decode z* into a robot and export it as MJCF

The decoder outputs the right half (torso + neck + right arm, 120-D). It is mirrored to a full upper
body, joints with ‖ω‖ < ε = 0.5 are deactivated, and a MuJoCo model is built from capsules and
spheres. Because the model uses only primitive geoms, the exported XML is a **single
self-contained file**: load it directly in MuJoCo, or use it as the kinematic blueprint for a
CAD/hardware design (paper §V-E).

In [ ]:
with th.no_grad():
    decoded = ae.decode(th.tensor(z_star[None, :], dtype=th.float32)).numpy()   # (1, 120)
xml_dict = _build_envs_from_decoded(rc, decoded, repr_kind)      # {'0': <mjcf string>}
design_xml = next(iter(xml_dict.values()))

env = MuJoCoParserClass(name='lego_design', xml_string=design_xml, verbose=False)
print(f'active joints: {env.n_joint}')
for jn in env.joint_names:
    print('  ', jn)

# Sanity check: the XML is standalone (no external mesh/texture files)
assert 'file=' not in design_xml
mujoco.MjModel.from_xml_string(design_xml)
print(f'\nMJCF: {len(design_xml)} chars, loads standalone ✓')

if SAVE_OUTPUTS:
    out_xml = os.path.join(TUTORIAL_OUT, f'lego_design_{motion_label}.xml')
    with open(out_xml, 'w') as f:
        f.write(design_xml)
    print(f'Saved: {out_xml}')
else:
    print('SAVE_OUTPUTS=False: not writing the MJCF. First lines:')
    print('\n'.join(design_xml.splitlines()[:12]))

## 3. Retarget the motion and recompute the objective

`directional_preretarget_postrigging` maps the human joints-of-interest (JOI: spine, neck,
shoulders, elbows, wrists) to the matching robot bodies via scaled direction vectors and solves
IK frame by frame (paper §III-C). The objective is Eq. (7):
PA-MPJPE (mm, per-frame Procrustes alignment) + λ_joint·N_joint with λ_joint = 3.5.
For the paper designs this reproduces the numbers in `reference_objective`.

In [ ]:
override = override_joi(env, repr_kind)                  # robot body for each JOI
joi_body = get_joi_body_name_of_irae(override=override)
joi_keys = list(joi_body.keys())                          # ['root','base','spine','neck','rs',...]
print('JOI → robot body:', {k: v for k, v in override.items()})

if MOTION_SOURCE == 'smpl':
    secs, chains = get_chains_from_smpl(
        file_path=motion_path, smpl=SMPLModelClass(model_path=SMPL_MODEL),
        intp_traj=False, plot_chain_graph=False)
    human_pos = get_smpl_pos(chains, joi_body)            # (L, J, 3), 'root' excluded
else:
    secs, chains = get_chains_from_bvh_cmu_mocap(
        bvh_path=motion_path, p_rate=0.056444, zup=True, plot_chain_graph=False, verbose=False)
    human_pos = get_cmu_pos(chains, joi_body)             # (L, J, 3), 'root' excluded
L = len(chains); HZ = int(round(1 / (secs[1] - secs[0])))
print(f'motion: {L} frames @ {HZ} Hz')

t0 = time.time()
qpos_list = directional_preretarget_postrigging(
    env, secs, chains,
    ik_p_spine=True, ik_p_neck=False, leg_fixed=True,
    src_name=MOTION_SOURCE, trgt_name='irae', joint_names_for_ik=None,
    max_ik_tick=100, ik_stepsize=1, ik_err_th=1e-3, print_ik_failure=False,
    ik_R_hand=False, ik_R_foot=False, reset_env=True, animate=False,
    plot_joi_src=False, plot_joi_trgt=False, render_every=1, verbose=False,
    print_every=30, pause_each_tick=False, post_rigging=True, override=override)
print(f'retargeted in {time.time() - t0:.1f}s')

# Robot JOI positions along the retargeted motion
robot_names = [joi_body[k] for k in joi_keys[1:]]          # same order as human_pos
robot_pos = np.zeros_like(human_pos)
env.reset(step=True)
for t in range(L):
    env.forward(q=qpos_list[t])
    for j, b in enumerate(robot_names):
        robot_pos[t, j] = env.data.body(b).xpos

pa_mpjpe  = _sequence_pa_mpjpe(human_pos, robot_pos, with_scale=True) * 1000
objective = pa_mpjpe + 3.5 * env.n_joint
print(f'\nPA-MPJPE  = {pa_mpjpe:.2f} mm')
print(f'objective = {pa_mpjpe:.2f} + 3.5 × {env.n_joint} joints = {objective:.4f}')
if DESIGN_LATENT is None and MOTION_SOURCE == 'cmu' and CUSTOM_BVH is None:
    print(f'paper reference: {reference_objective[MOTION]:.4f}')

## 4. Human vs. robot: key poses and hand trajectories

Left: key frames with the robot's JOI skeleton aligned to the human (per-frame similarity
transform, exactly what PA-MPJPE measures). Right: hand trajectories over the whole motion after a
single sequence-level alignment: the analogue of the red/blue curves in paper Fig. 5.

In [ ]:
idx  = {k: i for i, k in enumerate(joi_keys[1:])}
EDGES = [('base','spine'), ('spine','neck'),
         ('neck','rs'), ('rs','re'), ('re','rw'),
         ('neck','ls'), ('ls','le'), ('le','lw')]

def draw_skeleton(ax, P, color, lw=2, label=None):
    for n, (a, b) in enumerate(EDGES):
        seg = P[[idx[a], idx[b]]]
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], '-o', color=color, lw=lw, ms=3,
                label=label if n == 0 else None)

def set_equal_3d(ax, P):
    c = P.reshape(-1, 3).mean(0); r = np.abs(P.reshape(-1, 3) - c).max()
    ax.set_xlim(c[0]-r, c[0]+r); ax.set_ylim(c[1]-r, c[1]+r); ax.set_zlim(c[2]-r, c[2]+r)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

key_frames = np.linspace(0, L - 1, 6).astype(int)
fig = plt.figure(figsize=(20, 4.2))
for n, t in enumerate(key_frames):
    ax = fig.add_subplot(1, len(key_frames), n + 1, projection='3d')
    aligned, _, _, _ = _similarity_transform_umeyama(robot_pos[t], human_pos[t], with_scale=True)
    draw_skeleton(ax, human_pos[t], 'tab:gray', lw=3, label='human')
    draw_skeleton(ax, aligned, 'tab:red', label='robot')
    set_equal_3d(ax, np.concatenate([human_pos[t], aligned]))
    ax.view_init(elev=10, azim=-60); ax.set_title(f'frame {t}')
    if n == 0: ax.legend(loc='upper left', fontsize=8)
fig.suptitle('Fig 5.2: key poses (gray: human, red: robot, Procrustes-aligned per frame)')
plt.tight_layout(); plt.show()

# Sequence-level alignment of all robot JOI points onto the human
_, s, R, tvec = _similarity_transform_umeyama(robot_pos.reshape(-1, 3), human_pos.reshape(-1, 3))
robot_seq = (s * (R @ robot_pos.reshape(-1, 3).T) + tvec).T.reshape(robot_pos.shape)

fig = plt.figure(figsize=(12, 5))
for n, (hand, title) in enumerate([('rw', 'right hand'), ('lw', 'left hand')]):
    ax = fig.add_subplot(1, 2, n + 1, projection='3d')
    ax.plot(*human_pos[:, idx[hand]].T, color='tab:blue', lw=2, label='human')
    ax.plot(*robot_seq[:, idx[hand]].T, color='tab:red', lw=2, label='robot')
    draw_skeleton(ax, human_pos[0], 'lightgray', lw=1)
    set_equal_3d(ax, np.concatenate([human_pos, robot_seq]))
    ax.view_init(elev=10, azim=-60); ax.set_title(title); ax.legend()
fig.suptitle('Fig 5.3: hand trajectories (blue: human, red: robot)')
plt.tight_layout(); plt.show()

## 5. Render the robot executing the motion

Offscreen rendering with `mujoco.Renderer` (no window is opened). On a headless machine, start
Jupyter with `MUJOCO_GL=osmesa` (or `egl`), or under `xvfb-run -a`. The camera is framed automatically from the robot's
workspace over the whole motion.

In [ ]:
# Frame the camera from all body positions over the motion
env.reset(step=True)
pts = []
for t in range(0, L, max(1, L // 30)):
    env.forward(q=qpos_list[t]); pts.append(env.data.xpos[1:].copy())
pts = np.concatenate(pts)
center, extent = (pts.max(0) + pts.min(0)) / 2, (pts.max(0) - pts.min(0)).max()

cam = mujoco.MjvCamera()
cam.lookat[:] = center
cam.distance  = 1.5 * extent
cam.azimuth, cam.elevation = 160, -10
renderer = mujoco.Renderer(env.model, height=480, width=480)

STRIDE = 2                       # render every 2nd frame
frames = []
for t in range(0, L, STRIDE):
    env.forward(q=qpos_list[t])
    renderer.update_scene(env.data, camera=cam)
    frames.append(renderer.render().copy())
renderer.close()
print(f'rendered {len(frames)} frames')

fig, axes = plt.subplots(1, 6, figsize=(20, 3.6))
for ax, t in zip(axes, key_frames):
    ax.imshow(frames[min(t // STRIDE, len(frames) - 1)]); ax.set_title(f'frame {t}'); ax.axis('off')
fig.suptitle(f'Fig 5.4: designed robot executing the {motion_label} motion')
plt.tight_layout(); plt.show()

if SAVE_OUTPUTS:
    import cv2
    out_mp4 = os.path.join(TUTORIAL_OUT, f'lego_design_{motion_label}.mp4')
    vw = cv2.VideoWriter(out_mp4, cv2.VideoWriter_fourcc(*'mp4v'), max(1, HZ // STRIDE),
                         (frames[0].shape[1], frames[0].shape[0]))
    for f in frames:
        vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    vw.release()
    fig.savefig(os.path.join(TUTORIAL_OUT, f'lego_design_{motion_label}_frames.png'), dpi=120, bbox_inches='tight')
    print(f'Saved: {out_mp4}')
else:
    print('SAVE_OUTPUTS=False: not writing the mp4.')

## 6. Using your own motion

Any BVH with a **CMU-style skeleton** works. The retargeter reads these joints:

| JOI | BVH joint |
|---|---|
| base | midpoint of `LeftUpLeg` / `RightUpLeg` |
| spine, neck | `Spine`, `Neck` |
| shoulders, elbows, hands | `RightArm`/`LeftArm`, `RightForeArm`/`LeftForeArm`, `RightHand`/`LeftHand` |

To use it:
1. Check the file with the cell below.
2. Set `CUSTOM_BVH = '/path/to/motion.bvh'` in §1 and re-run §3-§5 to evaluate a design on it.
3. To *optimize* a design for it, put the path in `MOTIONS_BVH` in
   `02_design_optimization_voo.ipynb` and run the VOO demo; pass the best latent back here as
   `DESIGN_LATENT`.

`p_rate=0.056444` converts CMU's inch-based units to meters; adjust it if your BVH uses another scale.

In [ ]:
REQUIRED = ['Hips', 'Spine', 'Neck', 'LeftUpLeg', 'RightUpLeg',
            'RightArm', 'RightForeArm', 'RightHand', 'LeftArm', 'LeftForeArm', 'LeftHand']

def check_bvh(path):
    s, ch = get_chains_from_bvh_cmu_mocap(bvh_path=path, p_rate=0.056444, zup=True,
                                          plot_chain_graph=False, verbose=False)
    missing = [j for j in REQUIRED if j not in ch[0].joint_names]
    print(f'{os.path.basename(path)}: {len(ch)} frames @ {1 / (s[1] - s[0]):.0f} Hz')
    print('  OK: compatible with the retargeter' if not missing else f'  missing joints: {missing}')
    return not missing

check_bvh(bvh_path)

## 7. Using an SMPL motion

The retargeter also takes SMPL motions, such as text-to-motion results of
[MDM](https://github.com/GuyTevet/motion-diffusion-model). Retargeting, the objective and the
rendering are the same as for a BVH; only the loader changes (`get_chains_from_smpl`, `src_name='smpl'`).

Neither the SMPL model nor SMPL motions are included (license / size). To use them:
1. **SMPL model**: register at https://smpl.is.tue.mpg.de/, download the model, convert it with
   [`preprocess.py`](https://github.com/CalciferZh/SMPL) (needs `chumpy`) and place the output at
   `data/smpl_model/model.pkl` (the default `SMPL_MODEL` in §1; paths there are relative to `notebook/ae_test/`).
2. **Motion**: generate a motion with MDM and export its SMPL parameters with
   `python -m visualize.render_mesh --input_path <sample.mp4>`, which writes `*_smpl_params.npy`.
3. Put the file at the default `SMPL_MOTION` path in §1 (or point `SMPL_MOTION` at it), set
   `MOTION_SOURCE = 'smpl'` and re-run §1-§5. With the files missing, §1 stops with a pointer here.
4. To *optimize* a design for it, set the same `MOTION_SOURCE` / `SMPL_MOTION` in
   `02_design_optimization_voo.ipynb` and run the VOO demo; pass the best latent back here as
   `DESIGN_LATENT`.

The cell below checks both files and is skipped while they are missing.


In [ ]:
def check_smpl(npy_path, model_path=SMPL_MODEL):
    for p in (model_path, npy_path):
        if p is None or not os.path.exists(p):
            print(f'not found: {p} (see the steps above)')
            return False
    s, ch = get_chains_from_smpl(file_path=npy_path, smpl=SMPLModelClass(model_path=model_path),
                                 intp_traj=False, plot_chain_graph=False)
    print(f'{os.path.basename(npy_path)}: {len(ch)} frames @ {1 / (s[1] - s[0]):.0f} Hz')
    return True

check_smpl(SMPL_MOTION)

---

## Summary

- Decoded an optimized latent z* into a full upper-body kinematic structure
- Exported it as a standalone MJCF (`lego_design_<motion>.xml` with `SAVE_OUTPUTS=True`)
- Retargeted the target motion and reproduced the paper's objective (PA-MPJPE + λ·N_joint)
- Compared human and robot poses / hand trajectories, and rendered the robot in motion
- Saw how to swap the BVH for an SMPL motion (§7)

The physical prototypes in paper Fig. 4-5 were built from these decoded structures (scaled and
lightly adjusted for assembly); `asset/lego_{easy,middle,hard}/` contain those as-built models,
and `notebook/ae_test/mr_lego_*_with_postrigging.ipynb` retargets the motions onto them.

**Related tutorials**:
- [`02_design_optimization_voo.ipynb`](./02_design_optimization_voo.ipynb): find z* for a motion
- [`04_latent_traversal.ipynb`](./04_latent_traversal.ipynb): morph between designs